# Análise dos Benchmarks de Multiplicação Distribuída

Este notebook lê o arquivo `benchmark_results.csv` gerado pelo `Client.py`, compara os tempos de execução serial e distribuída/paralela, calcula o speedup e exibe uma prévia das matrizes da última execução.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8")
%matplotlib inline

CSV_PATH = Path("benchmark_results.csv")
MATRICES_PATH = Path("last_run_matrices.json")

## 1. Carregamento dos resultados

Execute primeiro os servidores e depois o cliente. O cliente adiciona uma linha neste CSV a cada tamanho de matriz testado.

In [ ]:
expected_columns = [
    "matrix_size",
    "serial_time_ms",
    "parallel_time_ms",
    "speedup",
    "num_servers",
    "timestamp",
]

if not CSV_PATH.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {CSV_PATH.resolve()}")

df = pd.read_csv(CSV_PATH)

missing = set(expected_columns) - set(df.columns)
if missing:
    raise ValueError(f"O CSV não possui as colunas esperadas: {sorted(missing)}")

df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
df = df.sort_values(["matrix_size", "timestamp"]).reset_index(drop=True)

df

## 2. Média dos resultados por tamanho

Se você rodar o cliente várias vezes, esta tabela consolida a média por tamanho de matriz e quantidade de servidores.

In [ ]:
if df.empty:
    print("O CSV ainda não possui resultados. Execute o Client.py para preencher o benchmark.")
    summary = df.copy()
else:
    summary = (
        df.groupby(["matrix_size", "num_servers"], as_index=False)
        .agg(
            serial_time_ms=("serial_time_ms", "mean"),
            parallel_time_ms=("parallel_time_ms", "mean"),
            runs=("matrix_size", "count"),
        )
        .sort_values("matrix_size")
    )
    summary["speedup"] = summary["serial_time_ms"] / summary["parallel_time_ms"]

summary

## 3. Gráficos comparativos

Esta seção mostra dois gráficos comparativos e uma tabela visual do speedup. Na tabela, valores com speedup maior que `1.0` aparecem em verde; valores menores ou iguais a `1.0` aparecem em vermelho.

In [ ]:
if summary.empty:
    print("Sem dados para plotar.")
else:
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(summary["matrix_size"], summary["serial_time_ms"], marker="o", label="Serial")
    ax.plot(summary["matrix_size"], summary["parallel_time_ms"], marker="o", label="Distribuído/paralelo")
    ax.set_title("Tempo de execução por tamanho de matriz")
    ax.set_xlabel("Tamanho da matriz (n x n)")
    ax.set_ylabel("Tempo médio (ms)")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(summary["matrix_size"].astype(str), summary["speedup"])
    ax.axhline(1, color="black", linewidth=1, linestyle="--")
    ax.set_title("Speedup da execução distribuída")
    ax.set_xlabel("Tamanho da matriz (n x n)")
    ax.set_ylabel("Speedup = tempo serial / tempo distribuído")
    ax.grid(True, axis="y", alpha=0.3)
    plt.show()

In [ ]:
if summary.empty:
    print("Sem dados para montar a tabela de speedup.")
else:
    table_df = summary[["matrix_size", "speedup", "num_servers", "runs"]].copy()
    table_df["Tamanho"] = table_df["matrix_size"].apply(lambda n: f"{int(n)}x{int(n)}")
    table_df["Speedup"] = table_df["speedup"].apply(lambda value: f"{value:.2f}x")
    table_df["Servidores"] = table_df["num_servers"].astype(int).astype(str)
    table_df["Execuções"] = table_df["runs"].astype(int).astype(str)

    display_columns = ["Tamanho", "Speedup", "Servidores", "Execuções"]
    table_data = table_df[display_columns].values.tolist()

    cell_colors = []
    for row in table_df.itertuples(index=False):
        speedup_color = "#b7e4c7" if row.speedup > 1.0 else "#f8c8c8"
        cell_colors.append(["#f5f5f5", speedup_color, "#f5f5f5", "#f5f5f5"])

    fig, ax = plt.subplots(figsize=(8, max(2.5, 0.55 * len(table_data) + 1.5)))
    ax.axis("off")
    table = ax.table(
        cellText=table_data,
        colLabels=display_columns,
        cellColours=cell_colors,
        colColours=["#d9e2ec"] * len(display_columns),
        cellLoc="center",
        loc="center",
    )
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    table.scale(1, 1.5)
    ax.set_title("Tabela visual de speedup por tamanho de matriz", pad=20)
    plt.show()

## 4. Exibição das Matrizes Geradas

Esta seção lê o arquivo auxiliar `last_run_matrices.json`, gerado pelo `Client.py` a cada execução, e exibe as primeiras `5x5` posições das matrizes `A`, `B`, dos resultados parciais de cada servidor e da matriz final `C`.

In [ ]:
def preview_matrix(matrix_data, rows=5, cols=5):
    matrix = np.array(matrix_data)
    return matrix[:rows, :cols]


if not MATRICES_PATH.exists():
    print("Arquivo last_run_matrices.json não encontrado. Execute o Client.py primeiro.")
else:
    with MATRICES_PATH.open("r", encoding="utf-8") as json_file:
        last_run = json.load(json_file)

    print(f"Última execução: {last_run.get('timestamp', 'sem timestamp')}")
    print(f"Servidores: {', '.join(last_run.get('servers', []))}\n")

    matrices = last_run.get("matrices", {})
    for size_key in sorted(matrices, key=lambda value: int(value)):
        item = matrices[size_key]
        n = int(item["matrix_size"])

        print(f"Tamanho {n}x{n}")
        print("  Matriz A (primeiras 5x5):")
        print(preview_matrix(item["A"]))
        print("  Matriz B (primeiras 5x5):")
        print(preview_matrix(item["B"]))
        for partial in item.get("partial_results", []):
            server_number = partial.get("server", "?")
            rows = partial.get("rows", "?")
            cols = partial.get("cols", "?")
            print(f"  Resultado parcial do Servidor {server_number} ({rows}x{cols}) - primeiras 5x5:")
            print(preview_matrix(partial["matrix"]))

        print("  Matriz C final (primeiras 5x5):")
        print(preview_matrix(item["C"]))
        print()

## 5. Interpretação

Em matrizes pequenas, o custo de comunicação via socket e serialização pode ser maior que o ganho de paralelismo. Em matrizes maiores, o cálculo tende a dominar o tempo total e o speedup costuma melhorar, desde que os servidores tenham recursos suficientes.

In [ ]:
if summary.empty:
    print("Ainda não há resultados para analisar.")
else:
    for row in summary.itertuples(index=False):
        print(
            f"Matriz {int(row.matrix_size)}x{int(row.matrix_size)} com {int(row.num_servers)} servidores: "
            f"serial={row.serial_time_ms:.2f} ms, "
            f"distribuído={row.parallel_time_ms:.2f} ms, "
            f"speedup={row.speedup:.2f}x ({int(row.runs)} execução/execuções)."
        )

    best = summary.loc[summary["speedup"].idxmax()]
    print(
        "\nMelhor speedup observado: "
        f"{best['speedup']:.2f}x para matriz {int(best['matrix_size'])}x{int(best['matrix_size'])}."
    )